#identificação de fraudes#

1# review em dias que o restaurante está fechado (#**eliminar**#)

In [0]:
from pyspark.sql.functions import col, dayofweek, when

# Carrega reviews da camada silver
reviews = spark.table("workspace.yelp_ing.silver_review")

# Carrega business para acessar horários
business = spark.table("workspace.yelp_ing.silver_business")

# Adiciona coluna com o dia da semana da review (1=Domingo, 2=Segunda, ..., 7=Sábado)
reviews_with_weekday = reviews.withColumn("review_weekday", dayofweek(col("date")))

# Mapeia dia da semana numérico para nome do campo no struct hours
reviews_with_day_name = reviews_with_weekday.withColumn(
    "day_name",
    when(col("review_weekday") == 1, "Sunday")
    .when(col("review_weekday") == 2, "Monday")
    .when(col("review_weekday") == 3, "Tuesday")
    .when(col("review_weekday") == 4, "Wednesday")
    .when(col("review_weekday") == 5, "Thursday")
    .when(col("review_weekday") == 6, "Friday")
    .when(col("review_weekday") == 7, "Saturday")
)

# Junta com business para acessar os horários
reviews_joined = reviews_with_day_name.join(
    business.select("business_id", "hours", "is_open", "name"),
    "business_id",
    "left"
)

# Extrai o horário específico do dia da semana
from pyspark.sql.functions import expr

# Adiciona coluna 'day_hours' com o horário de funcionamento do restaurante no dia da semana da review
reviews_with_hours = reviews_joined.withColumn(
    "day_hours",
    expr("CASE " +
         "WHEN day_name = 'Sunday' THEN hours.Sunday " +
         "WHEN day_name = 'Monday' THEN hours.Monday " +
         "WHEN day_name = 'Tuesday' THEN hours.Tuesday " +
         "WHEN day_name = 'Wednesday' THEN hours.Wednesday " +
         "WHEN day_name = 'Thursday' THEN hours.Thursday " +
         "WHEN day_name = 'Friday' THEN hours.Friday " +
         "WHEN day_name = 'Saturday' THEN hours.Saturday " +
         "END")
)
)

# Filtra reviews feitas em dias que o restaurante estava fechado (day_hours é null)
fraud_reviews = reviews_with_hours.filter(col("day_hours").isNull() & col("hours").isNotNull())

display(fraud_reviews.select("review_id", "business_id", "name", "date", "day_name", "day_hours", "stars"))

2# pico de reviews (suposta compra)

In [0]:
from pyspark.sql.functions import col, count, avg, month, dayofweek

# Adiciona colunas de mês e dia da semana
reviews_with_time = reviews.withColumn("review_month", month(col("date"))) \
                           .withColumn("review_weekday", dayofweek(col("date")))

# Conta reviews por restaurante e data
daily_counts = reviews_with_time.groupBy("business_id", "date", "review_month", "review_weekday") \
    .agg(count("*").alias("daily_review_count"))

# Calcula média de reviews por restaurante, mês e dia da semana
avg_reviews = daily_counts.groupBy("business_id", "review_month", "review_weekday") \
    .agg(avg("daily_review_count").alias("avg_review_count"))

# Junta contagem diária com média
daily_with_avg = daily_counts.join(
    avg_reviews,
    ["business_id", "review_month", "review_weekday"],
    "left"
)

# Define threshold (exemplo: 3x a média)
threshold = 3

# Filtra dias com reviews muito acima da média
high_review_days = daily_with_avg.filter(col("daily_review_count") > threshold * col("avg_review_count"))

display(high_review_days)

3# comportamentos atípicos de reviews

In [0]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col, avg, stddev, count, sum, when

# Calcula métricas por restaurante
metrics = reviews.groupBy("business_id").agg(
    count("*").alias("review_count"),
    avg("stars").alias("avg_rating"),
    stddev("stars").alias("std_rating"),
    sum(when(col("stars") == 5, 1).otherwise(0)).alias("count_5star")
)

# Monta vetor de features
assembler = VectorAssembler(
    inputCols=["review_count", "avg_rating", "std_rating", "count_5star"],
    outputCol="features"
)
metrics_vector = assembler.transform(metrics)

# Aplica KMeans clustering
kmeans = KMeans(featuresCol="features", k=4, seed=42)
model = kmeans.fit(metrics_vector)
clusters = model.transform(metrics_vector)

# Detecta outliers baseado em média e desvio padrão
stats = metrics.select(
    avg("review_count").alias("mean_count"),
    stddev("review_count").alias("std_count"),
    avg("avg_rating").alias("mean_rating"),
    stddev("avg_rating").alias("std_rating"),
    avg("count_5star").alias("mean_5star"),
    stddev("count_5star").alias("std_5star")
).collect()[0]

# Define limites para outliers (exemplo: > média + 2*desvio padrão)
outliers = clusters.filter(
    (col("review_count") > stats.mean_count + 2 * stats.std_count) |
    (col("avg_rating") > stats.mean_rating + 2 * stats.std_rating) |
    (col("count_5star") > stats.mean_5star + 2 * stats.std_5star)
)

# Ranqueia suspeitos por soma de z-scores das métricas
from pyspark.sql.functions import abs

suspects = outliers.withColumn("z_review_count", abs((col("review_count") - stats.mean_count) / stats.std_count)) \
    .withColumn("z_avg_rating", abs((col("avg_rating") - stats.mean_rating) / stats.std_rating)) \
    .withColumn("z_count_5star", abs((col("count_5star") - stats.mean_5star) / stats.std_5star)) \
    .withColumn("suspicion_score", col("z_review_count") + col("z_avg_rating") + col("z_count_5star")) \
    .orderBy(col("suspicion_score").desc())

display(suspects)

4# analise de similaridade de texto entre reviews (copiar e colar)

In [0]:
from pyspark.sql.functions import col
from pyspark.ml.feature import Tokenizer, HashingTF, IDF
from pyspark.ml.feature import MinHashLSH

# Carrega reviews
silver_reviews = spark.table("workspace.yelp_ing.silver_review")

# Tokeniza texto
tokenizer = Tokenizer(inputCol="text", outputCol="words")
tokenized = tokenizer.transform(silver_reviews)

# TF
hashingTF = HashingTF(inputCol="words", outputCol="rawFeatures", numFeatures=1000)
featurized = hashingTF.transform(tokenized)

# IDF
idf = IDF(inputCol="rawFeatures", outputCol="features")
idfModel = idf.fit(featurized)
rescaled = idfModel.transform(featurized)

# MinHashLSH para similaridade
mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=5)
model = mh.fit(rescaled)
similar_pairs = model.approxSimilarityJoin(rescaled, rescaled, 0.2, "distance") \
    .filter(col("datasetA.review_id") < col("datasetB.review_id")) \
    .select(
        col("datasetA.review_id").alias("review_id_A"),
        col("datasetA.user_id").alias("user_id_A"),
        col("datasetA.business_id").alias("restaurant_id_A"),
        col("datasetB.review_id").alias("review_id_B"),
        col("datasetB.user_id").alias("user_id_B"),
        col("datasetB.business_id").alias("restaurant_id_B"),
        col("distance")
    )

display(similar_pairs)

# Soma quantidade de comentários parecidos por review
from pyspark.sql.functions import count

similar_count = similar_pairs.groupBy("review_id_A", "user_id_A", "restaurant_id_A") \
    .agg(count("review_id_B").alias("similar_comment_count")) \
    .orderBy(col("similar_comment_count").desc())

display(similar_count)